In [35]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

In [40]:
data = pd.read_csv("IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [41]:
def remove_HTML(text):
  text = re.sub(r"<.*?>" , "",text)
  return text

def remove_url(text):
  text = re.sub(r"http\S+" , "" , text)
  return text



In [42]:
data['review'] = data['review'].apply(remove_HTML)
data['review'] = data['review'].apply(remove_url)

le = LabelEncoder()
data['sentiment'] = le.fit_transform(data['sentiment'])

In [43]:
X = data['review']
y = data['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_Val, y_train, y_Val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

In [44]:
# A Hugging Face Arrow table is the underlying Apache Arrow data structure used by the datasets library to store, cache, and process machine learning data efficiently.
# Dataset.from_pandas(...): Converts the DataFrame into a memory-efficient Hugging Face Arrow table.
# [["review", "label"]]: Filters the DataFrame to keep only the text feature (review) and the target label (label).
# preserve_index=False: Prevents pandas row indices from being stored as an extra column (like __index_level_0__) in your final dataset, keeping it clean.


train_DataSet = Dataset.from_pandas(pd.DataFrame({"review": X_train, "label": y_train}), preserve_index=False)
val_DataSet = Dataset.from_pandas(pd.DataFrame({"review": X_Val, "label": y_Val}), preserve_index=False)
test_DataSet = Dataset.from_pandas(pd.DataFrame({"review": X_test, "label": y_test}), preserve_index=False)

In [45]:

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["review"], truncation=True, max_length=256)

train_DataSet = train_DataSet.map(tokenize , batched=True )
val_DataSet = val_DataSet.map(tokenize , batched=True )
test_DataSet = test_DataSet.map(tokenize , batched=True )

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [47]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

args = TrainingArguments(
    output_dir="distilbert-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
)

In [51]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_DataSet,
    eval_dataset=val_DataSet,
    processing_class=tokenizer,  # Replaces 'tokenizer' parameter
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.224712,0.211118,0.913800,0.913989
2,0.147686,0.258446,0.917400,0.917614


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5626, training_loss=0.2086682989213376, metrics={'train_runtime': 678.9477, 'train_samples_per_second': 132.558, 'train_steps_per_second': 8.286, 'total_flos': 5960676933386112.0, 'train_loss': 0.2086682989213376, 'epoch': 2.0})

In [52]:
results = trainer.evaluate(test_DataSet)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.147686,0.174174,2,0.932900,0.933147


In [53]:
trainer.save_model("my_sentiment_model")
tokenizer.save_pretrained("my_sentiment_model")

from transformers import pipeline
clf = pipeline("sentiment-analysis", model="my_sentiment_model")

print(clf("This movie was absolutely fantastic, I loved every minute!"))
print(clf("Not good. The plot made no sense and it was boring."))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.995503842830658}]
[{'label': 'NEGATIVE', 'score': 0.9943460822105408}]


In [55]:
import shutil
from google.colab import files

shutil.make_archive("my_sentiment_model" , "zip" ,"my_sentiment_model" )

files.download("my_sentiment_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>